# mFRR price dashboard with weather data

This dashboard has a start-date and end-date chooser.

In [ ]:
# Import the packages we need
from datetime import timedelta

import ipywidgets as widgets
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

## 1. Load the prepared dataset

In [ ]:
# Use the merged dataset from the data-preparation notebook
dashboard_data = pd.read_csv('combined_mfrr_weather.csv')
dashboard_data['TimeUTC'] = pd.to_datetime(dashboard_data['TimeUTC'], utc=True)
dashboard_data['TimeDK'] = dashboard_data['TimeUTC'].dt.tz_convert('Europe/Copenhagen')
# Lets check if its looks right: 
dashboard_data.head()

## 2. Build the dashboard

Lets build the dashboard with inspiration from this website: https://thelinuxcode.com/interactive-controls-in-jupyter-notebooks-build-fast-insightful-exploration-loops/

### Notes about the wind scenarios

- The wind threshold is selected by the user and is measured in m/s.
- **Low wind** contains observations with wind speed less than or equal to the threshold.
- **High wind** contains observations with wind speed above the threshold.
- The boxplots use only observations inside the selected date period.
- The line inside each box is the median. The box represents the middle 50% of prices, while points outside the whiskers are shown as outliers.
- The maximum-price control affects only the CM-price time series. The boxplots retain the complete price distribution.

In [ ]:
# Define the start and end date pickers, the update button, and the output area
start_date = widgets.DatePicker(
    description='From',
    value=dashboard_data['TimeDK'].max().date() - timedelta(days=30),
    min=dashboard_data['TimeDK'].min().date(),
    max=dashboard_data['TimeDK'].max().date(),
)
end_date = widgets.DatePicker(
    description='Through',
    value=dashboard_data['TimeDK'].max().date(),
    min=dashboard_data['TimeDK'].min().date(),
    max=dashboard_data['TimeDK'].max().date(),
)
# I have also seen extreme mFRR-prices, so let me get another widget to set the y-axis limit for the mFRR-prices: 
max_price = widgets.FloatText(
    description='Max price:',
    value=round(float(dashboard_data['UpPriceDKK'].max()), 2),
    min=0,
    step=50,
)
wind_threshold = widgets.BoundedFloatText(
    description='Wind threshold:',
    value=round(float(dashboard_data['WindSpeed10mMeanMS'].median()), 2),
    min=0,
    max=round(float(dashboard_data['WindSpeed10mMeanMS'].max()), 2),
    step=0.5,
)

update = widgets.Button(description='Update dashboard')
output = widgets.Output()

# I define a function to show the dashboard based on the selected date range
def show_dashboard(_):
    # Through testing found out i need to clear output before plotting new figures. 
    output.clear_output()
    # This check also necessary due to my user testing. 
    if start_date.value is None or end_date.value is None or start_date.value > end_date.value:
        with output:
            print('Please choose a valid period.')
        return
    
    # If the date range is valid, proceed with plotting.
    selected_data = dashboard_data[
        (dashboard_data['TimeDK'] >= pd.Timestamp(start_date.value, tz='Europe/Copenhagen'))
        & (dashboard_data['TimeDK'] < pd.Timestamp(
            end_date.value + timedelta(days=1), tz='Europe/Copenhagen'
        ))
    ].copy()
    # Print a warning if my max price is lower than the maximum price in the selected period.
    if max_price.value < selected_data['UpPriceDKK'].max():
        with output:
            print(
                "Some price peaks are hidden."
            )
        
    low_wind_prices = selected_data.loc[
        selected_data['WindSpeed10mMeanMS'] <= wind_threshold.value,
        'UpPriceDKK',
    ]
    high_wind_prices = selected_data.loc[
        selected_data['WindSpeed10mMeanMS'] > wind_threshold.value,
        'UpPriceDKK',
    ]

    # The first three plots are time series. The fourth compares the two wind scenarios.
    figure, axes = plt.subplots(
        4, 1, figsize=(14, 12),
        gridspec_kw={'height_ratios': [2, 1, 1, 1.5]},
    )
    axes[1].sharex(axes[0])
    axes[2].sharex(axes[0])
    axes[0].tick_params(labelbottom=False)
    axes[1].tick_params(labelbottom=False)

# Plot the data for the four subplots:
    axes[0].plot(selected_data['TimeDK'], selected_data['UpPriceDKK'], color='black')
    axes[0].set(title='CM up price', ylabel='DKK/MW/h')
    # This line sets the y_limit: 
    axes[0].set_ylim(top=max_price.value)

    axes[1].plot(selected_data['TimeDK'], selected_data['WindSpeed10mMeanMS'], color='steelblue')
    axes[1].set(title='Average wind speed', ylabel='m/s')

    axes[2].plot(selected_data['TimeDK'], selected_data['SolarRadiationMeanWm2'], color='darkorange')
    axes[2].set(title='Average solar radiation', ylabel='W/m²', xlabel='Danish delivery time')
    # Important to make the boxplot here:
    axes[3].boxplot(
        [low_wind_prices, high_wind_prices],
        tick_labels=[
            f'Low wind\n≤ {wind_threshold.value:g} m/s',
            f'High wind\n> {wind_threshold.value:g} m/s',
        ],
        showfliers=True,
    )
    axes[3].set(
        title='mFRR-price distribution by wind scenario',
        ylabel='DKK/MW/h',
        )
    

    for axis in axes:
        axis.spines[['top', 'right']].set_visible(False)
        axis.grid(axis='y', color='lightgrey', linewidth=0.5)
# The x-axis is shared, so we only need to set the locator and formatter for the last subplot:
    axes[2].xaxis.set_major_locator(
        mdates.AutoDateLocator(minticks=5, maxticks=10, tz='Europe/Copenhagen')
    )
    axes[2].xaxis.set_major_formatter(
        mdates.DateFormatter('%d %b\n%H:%M', tz='Europe/Copenhagen')
    )
# Lets add a title to the figure and let the title be more on the left side
    figure.suptitle(
        f'mFRR CM price and weather: {start_date.value:%d %b %Y} to {end_date.value:%d %b %Y}',
        x=0.125, ha='left', fontsize=16
    )
    
    figure.tight_layout()
    with output:
        display(figure)
    plt.close(figure)

update.on_click(show_dashboard)
# There were in previous version some issues with the width of the widgets, lets try to fix it: 
for control in [start_date, end_date, max_price, wind_threshold, update]:
    control.layout.width = 'auto'
display(
    widgets.GridBox(
        [start_date, end_date, max_price, wind_threshold, update],
        layout=widgets.Layout(
            width='100%',
            grid_template_columns='repeat(3, minmax(0, 1fr))',
            grid_gap='12px',
            align_items='center',
        ),
    ),
    output,
)
show_dashboard(None)

GridBox(children=(DatePicker(value=datetime.date(2026, 9, 20), description='From', layout=Layout(width='auto')…

Output()